In [0]:
%python
SECRET_SCOPE = "default2"

tenant_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="tenant-id")
client_id = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-databricks-adls-appid")      
client_secret = dbutils.secrets.get(scope=SECRET_SCOPE, key="sp-databricks-adls-appkey")

storage_account = "dlspl21databricks"
container = "lechster10"

In [0]:
configs = {        # authentication data to connect
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

source_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
mount_point = f"/mnt/{container}"

In [0]:
# Mounts work by creating a local alias under the /mnt directory that stores the following information
# Location of the cloud object storage.
# Driver specifications to connect to the storage account or container.
# Security credentials required to access the data.

# impossible on shared cluster

try:
    dbutils.fs.mount(
        source=source_path,
        mount_point=mount_point,
        extra_configs=configs
    )
    print("Path has been mounted.")
except Exception as e:
    print(f"Mounting erro (it can already exist)")




Instead of mounting the container globally to the cluster (which is blocked on Shared clusters for security reasons), this method injects the Service Principal credentials directly into the active Spark session using a dictionary.

In [0]:
# just for checking

for key, value in configs.items():    
    spark_key = f"{key}.{storage_account}.dfs.core.windows.net"
    spark.conf.set(spark_key, value)

In [0]:
abfss_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

# 6. Test połączenia – sprawdzamy czy Spark "widzi" pliki
try:
    print(f"Reading from {abfss_path}")    
    files = dbutils.fs.ls(abfss_path) 
    print(files)

except Exception as e:
    print("No access")
   